In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.optim as optim


import import_ipynb
from common import assert_eq, assert_ge, check_eq, Patient, T # type: ignore
from cross_entropy import cross_entropy as _pure_cross_entropy # type: ignore


class _PureCrossEntropyLoss(nn.Module):
    def forward(self, input: tr.Tensor, target: tr.Tensor) -> tr.Tensor:
        return nn.BCELoss(reduction='mean')(input, nn.functional.one_hot(target).float())


def logistic_regression_nc_sgd_autograd(X: tr.Tensor, Y: tr.Tensor, epochs=2000, lr=0.1) -> tuple[float, callable]:
    """
    Perform logistic regression using Stochastic Gradient Descent (SGD) with autograd.

    Args:
        X: Input features of shape (Samples, Features)
        Y: Target values of shape (Samples, 1)
        epochs: Number of training epochs
        lr: Learning rate

    Returns:
        A tuple containing the final loss and a prediction function that takes new input data and returns predicted probabilities.
    """

    ((s, f), c) = (X.shape, len(set(Y.squeeze().tolist())))

    assert_eq(X.shape, (s, f))
    assert_eq(Y.shape, (s, 1))

    #
    # CrossEntropyLoss expects input logits as it combines a Softmax followed by 
    # the Cross Entropy as one operation, taking the advantage of the log-sum-exp 
    # trick is used for numerical stability.
    #
    model = nn.Linear(f, c)
    loss_fn = nn.CrossEntropyLoss()
    (weight, bias) = (model.weight, model.bias)
    
    #
    # For demonstation purposes only
    #
    # model = nn.Sequential(nn.Linear(f, c), nn.Softmax(dim=-1))
    # loss_fn = _PureCrossEntropyLoss()
    # (weight, bias) = (model[0].weight, model[0].bias)
    
    #
    # For demonstration purposes only
    #
    # model = nn.Sequential(nn.Linear(f, c), nn.LogSoftmax(dim=-1))
    # loss_fn = nn.NLLLoss()
    # (weight, bias) = (model[0].weight, model[0].bias)

    optimizer = optim.SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()

        Z = model(X)
        assert_eq(Z.shape, (s, c))

        L = loss_fn(Z, Y.squeeze().long())
        assert_eq(L.shape, ())
        
        L.backward()
        optimizer.step()

    return (L.item(), weight, bias)
    
    assert False, "Unsupported loss function"


def _test_logistic_regression_nc_sgd_autograd(epochs=2000, lr=0.1) -> None:
    """
    Test the logistic regression implementation on a synthetic dataset of patients. 
    The dataset consists of three classes: healthy (0), viral (1), and bacterial (2). 
    The model is trained on 70 samples and then evaluated on 30 samples (10 per class). 
    The test checks if the model achieves at least 90% accuracy on the evaluation set.
    """

    training_data = T([Patient([0.5, 0.25, 0.25]).data for _ in range(70)])

    X = training_data[:, :-1]
    X[:, 0] /= 100 # Temperature scaling
    X[:, 5] /= 200 # CRP scaling   

    Y = training_data[:, -1].unsqueeze(1)

    (_, model_W, model_b) = logistic_regression_nc_sgd_autograd(X, Y, epochs, lr)

    total = 0
    correct = 0

    for d in ([Patient([1.0, 0.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 1.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 0.0, 1.0]).data for _ in range(10)]):
        X = T(d[:-1])
        Y = T(d[-1])

        X[0] /= 100
        X[5] /= 200
        
        total += 1
        correct += check_eq((X @ model_W.t() + model_b).argmax(), Y)

    assert_ge(correct / total, 0.9)


if __name__ == "__main__":
    _test_logistic_regression_nc_sgd_autograd()